# SQL 面试与实习练习 Notebook

这份 notebook 按你给的要求重做了，覆盖：
- 基础查询：`SELECT` `WHERE` `ORDER BY` `LIMIT` `DISTINCT`
- 聚合与分组：`COUNT` `SUM` `AVG` `MAX` `MIN` `GROUP BY` `HAVING`
- 连表：`INNER JOIN` `LEFT JOIN` 多表 `JOIN`
- 其他基础：`NULL` 处理、子查询、`CASE WHEN`
- 进阶：条件聚合、去重统计、日期处理、`WITH`
- 窗口函数：`ROW_NUMBER()` `RANK()` `DENSE_RANK()` `SUM() OVER()` `AVG() OVER()`
- 业务题：用户分群、逾期率、转化率、分组表现、TopN、时间趋势

使用方式：
1. 先运行初始化数据库和辅助函数。
2. 每道题先写 `your_sql`。
3. 运行后看结果，再对照答案。

## 1. 初始化数据库

Goal：创建一套更接近实习场景的数据。

这次我们用 4 张表：
- `users`：用户信息
- `visits`：访问记录
- `orders`：下单记录
- `loans`：借款记录

In [ ]:
import sqlite3

print('SQLite version:', sqlite3.sqlite_version)

conn = sqlite3.connect(':memory:')
cur = conn.cursor()

setup_sql = '''
CREATE TABLE users (
    user_id INTEGER PRIMARY KEY,
    user_name TEXT,
    register_date TEXT,
    city TEXT,
    channel TEXT,
    segment TEXT
);

CREATE TABLE visits (
    visit_id INTEGER PRIMARY KEY,
    user_id INTEGER,
    visit_date TEXT,
    source TEXT
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    user_id INTEGER,
    order_date TEXT,
    order_amount REAL,
    product_type TEXT
);

CREATE TABLE loans (
    loan_id INTEGER PRIMARY KEY,
    user_id INTEGER,
    apply_date TEXT,
    due_date TEXT,
    repay_date TEXT,
    loan_amount REAL,
    status TEXT
);

INSERT INTO users VALUES
(1, 'Alice', '2024-01-02', 'Shanghai', 'organic', 'A'),
(2, 'Bob', '2024-01-05', 'Beijing', 'ad', 'A'),
(3, 'Cindy', '2024-01-10', 'Shanghai', 'ad', 'B'),
(4, 'David', '2024-01-15', 'Shenzhen', 'referral', NULL),
(5, 'Eric', '2024-02-01', 'Hangzhou', 'organic', 'B'),
(6, 'Frank', '2024-02-05', 'Suzhou', 'ad', 'C'),
(7, 'Grace', '2024-02-10', 'Beijing', 'referral', NULL),
(8, 'Helen', '2024-03-01', 'Shanghai', 'organic', 'A');

INSERT INTO visits VALUES
(101, 1, '2024-01-03', 'organic'),
(102, 2, '2024-01-06', 'ad'),
(103, 2, '2024-01-07', 'ad'),
(104, 3, '2024-01-11', 'ad'),
(105, 4, '2024-01-16', 'referral'),
(106, 5, '2024-02-02', 'organic'),
(107, 6, '2024-02-06', 'ad'),
(108, 6, '2024-02-07', 'ad'),
(109, 7, '2024-02-11', 'referral'),
(110, 8, '2024-03-02', 'organic');

INSERT INTO orders VALUES
(1001, 1, '2024-01-12', 300, 'electronics'),
(1002, 1, '2024-02-15', 500, 'grocery'),
(1003, 2, '2024-01-20', 800, 'electronics'),
(1004, 3, '2024-03-01', 200, 'beauty'),
(1005, 3, '2024-03-18', 900, 'electronics'),
(1006, 4, '2024-02-28', 400, 'home'),
(1007, 4, '2024-03-20', 700, 'home'),
(1008, 5, '2024-02-12', 150, NULL),
(1009, 6, '2024-03-05', 650, 'electronics'),
(1010, 6, '2024-03-25', 350, 'beauty');

INSERT INTO loans VALUES
(2001, 1, '2024-01-15', '2024-02-15', '2024-02-14', 1000, 'repaid'),
(2002, 2, '2024-01-25', '2024-02-25', '2024-03-02', 1500, 'overdue'),
(2003, 3, '2024-02-10', '2024-03-10', NULL, 1200, 'overdue'),
(2004, 1, '2024-03-05', '2024-04-05', '2024-04-03', 2000, 'repaid'),
(2005, 4, '2024-02-15', '2024-03-15', NULL, 1800, 'pending'),
(2006, 5, '2024-02-20', '2024-03-20', '2024-03-18', 900, 'repaid'),
(2007, 6, '2024-03-01', '2024-04-01', NULL, 1600, 'overdue'),
(2008, 8, '2024-03-10', '2024-04-10', NULL, 1100, 'pending');
'''

cur.executescript(setup_sql)
conn.commit()

print('数据库初始化完成。')

In [ ]:
def run_sql(sql, limit=10, show_sql=False):
    cursor = conn.execute(sql)
    columns = [desc[0] for desc in cursor.description] if cursor.description else []
    rows = cursor.fetchall()

    if show_sql:
        print('SQL:')
        print(sql.strip())
        print('=' * 100)

    if not columns:
        print('语句执行完成，没有返回结果集。')
        return

    print(f'共返回 {len(rows)} 行，当前最多显示前 {limit} 行。')
    print(' | '.join(columns))
    print('-' * 100)
    for row in rows[:limit]:
        print(' | '.join(str(value) for value in row))

    if len(rows) > limit:
        print(f'... 还有 {len(rows) - limit} 行未显示。')
        print('如果你想看更多，可以传入更大的 limit，比如 run_sql(sql, limit=30)。')

print('run_sql 已就绪。默认最多显示前 10 行，减少 notebook 渲染压力。')

## 2. 原始数据

In [7]:
run_sql('SELECT * FROM users;')

共返回 8 行，当前最多显示前 10 行。
user_id | user_name | register_date | city | channel | segment
----------------------------------------------------------------------------------------------------
1 | Alice | 2024-01-02 | Shanghai | organic | A
2 | Bob | 2024-01-05 | Beijing | ad | A
3 | Cindy | 2024-01-10 | Shanghai | ad | B
4 | David | 2024-01-15 | Shenzhen | referral | None
5 | Eric | 2024-02-01 | Hangzhou | organic | B
6 | Frank | 2024-02-05 | Suzhou | ad | C
7 | Grace | 2024-02-10 | Beijing | referral | None
8 | Helen | 2024-03-01 | Shanghai | organic | A


In [8]:
run_sql('SELECT * FROM visits;')

共返回 10 行，当前最多显示前 10 行。
visit_id | user_id | visit_date | source
----------------------------------------------------------------------------------------------------
101 | 1 | 2024-01-03 | organic
102 | 2 | 2024-01-06 | ad
103 | 2 | 2024-01-07 | ad
104 | 3 | 2024-01-11 | ad
105 | 4 | 2024-01-16 | referral
106 | 5 | 2024-02-02 | organic
107 | 6 | 2024-02-06 | ad
108 | 6 | 2024-02-07 | ad
109 | 7 | 2024-02-11 | referral
110 | 8 | 2024-03-02 | organic


In [9]:
run_sql('SELECT * FROM orders;')

共返回 10 行，当前最多显示前 10 行。
order_id | user_id | order_date | order_amount | product_type
----------------------------------------------------------------------------------------------------
1001 | 1 | 2024-01-12 | 300.0 | electronics
1002 | 1 | 2024-02-15 | 500.0 | grocery
1003 | 2 | 2024-01-20 | 800.0 | electronics
1004 | 3 | 2024-03-01 | 200.0 | beauty
1005 | 3 | 2024-03-18 | 900.0 | electronics
1006 | 4 | 2024-02-28 | 400.0 | home
1007 | 4 | 2024-03-20 | 700.0 | home
1008 | 5 | 2024-02-12 | 150.0 | None
1009 | 6 | 2024-03-05 | 650.0 | electronics
1010 | 6 | 2024-03-25 | 350.0 | beauty


In [10]:
run_sql('SELECT * FROM loans;')

共返回 8 行，当前最多显示前 10 行。
loan_id | user_id | apply_date | due_date | repay_date | loan_amount | status
----------------------------------------------------------------------------------------------------
2001 | 1 | 2024-01-15 | 2024-02-15 | 2024-02-14 | 1000.0 | repaid
2002 | 2 | 2024-01-25 | 2024-02-25 | 2024-03-02 | 1500.0 | overdue
2003 | 3 | 2024-02-10 | 2024-03-10 | None | 1200.0 | overdue
2004 | 1 | 2024-03-05 | 2024-04-05 | 2024-04-03 | 2000.0 | repaid
2005 | 4 | 2024-02-15 | 2024-03-15 | None | 1800.0 | pending
2006 | 5 | 2024-02-20 | 2024-03-20 | 2024-03-18 | 900.0 | repaid
2007 | 6 | 2024-03-01 | 2024-04-01 | None | 1600.0 | overdue
2008 | 8 | 2024-03-10 | 2024-04-10 | None | 1100.0 | pending


## 3. 基础查询

### 题目 1：`SELECT + WHERE + ORDER BY + LIMIT`

查询订单金额大于 400 的订单，按金额从高到低排序，只保留前 3 条。

In [ ]:
your_sql = '''
SELECT '请先把这里改成自己的 SQL' AS message;
'''
run_sql(your_sql)

In [ ]:
answer_sql = '''
SELECT *
FROM orders
WHERE order_amount > 400
ORDER BY order_amount DESC
LIMIT 3;
'''
run_sql(answer_sql)

### 题目 2：`DISTINCT`

查询所有不同的推广渠道。

In [ ]:
your_sql = '''
SELECT '请先把这里改成自己的 SQL' AS message;
'''
run_sql(your_sql)

In [ ]:
answer_sql = '''
SELECT DISTINCT channel
FROM users;
'''
run_sql(answer_sql)

### 题目 3：`NULL` 处理 + `CASE WHEN`

查询所有用户，若 `segment` 为空则显示为 `Unknown`；同时根据渠道把用户标记为 `paid` 或 `free`。

In [21]:
your_sql = '''
SELECT 
    user_id,
    user_name,
    COALESCE(segment,'Unknown') AS segment,
    CASE
        WHEN channel = 'ad' THEN 'paid'
        ELSE 'free'
    END AS channel_type
FROM users;
'''
run_sql(your_sql)

共返回 8 行，当前最多显示前 10 行。
user_id | user_name | segment | channel_type
----------------------------------------------------------------------------------------------------
1 | Alice | A | free
2 | Bob | A | paid
3 | Cindy | B | paid
4 | David | Unknown | free
5 | Eric | B | free
6 | Frank | C | paid
7 | Grace | Unknown | free
8 | Helen | A | free


In [20]:
answer_sql = '''
SELECT
    user_id,
    user_name,
    COALESCE(segment, 'Unknown') AS segment_label,
    CASE
        WHEN channel = 'ad' THEN 'paid'
        ELSE 'free'
    END AS channel_type
FROM users;
'''
run_sql(answer_sql)

共返回 8 行，当前最多显示前 10 行。
user_id | user_name | segment_label | channel_type
----------------------------------------------------------------------------------------------------
1 | Alice | A | free
2 | Bob | A | paid
3 | Cindy | B | paid
4 | David | Unknown | free
5 | Eric | B | free
6 | Frank | C | paid
7 | Grace | Unknown | free
8 | Helen | A | free


### 题目 4：聚合函数

统计订单表的订单数、总金额、平均金额、最大金额、最小金额。

In [26]:
your_sql = '''
SELECT 
COUNT(*) AS order_count,
SUM(order_amount) AS TOTAL_Amount,
AVG(order_amount) AS AVG_Amount,
MAX(order_amount),
MIN(order_amount)
FROM orders
'''
run_sql(your_sql)

共返回 1 行，当前最多显示前 10 行。
order_count | TOTAL_Amount | AVG_Amount | MAX(order_amount) | MIN(order_amount)
----------------------------------------------------------------------------------------------------
10 | 4950.0 | 495.0 | 900.0 | 150.0


In [ ]:
answer_sql = '''
SELECT
    COUNT(*) AS order_cnt,
    SUM(order_amount) AS total_amount,
    AVG(order_amount) AS avg_amount,
    MAX(order_amount) AS max_amount,
    MIN(order_amount) AS min_amount
FROM orders;
'''
run_sql(answer_sql)

## 4. 分组与聚合

### 题目 5：`GROUP BY`

按渠道统计注册用户数。

In [30]:
your_sql = '''
SELECT 
    COUNT(*) AS users_count,
    channel
FROM users
GROUP BY channel;
'''
run_sql(your_sql)

共返回 3 行，当前最多显示前 10 行。
users_count | channel
----------------------------------------------------------------------------------------------------
3 | ad
3 | organic
2 | referral


In [29]:
answer_sql = '''
SELECT
    channel,
    COUNT(*) AS user_cnt
FROM users
GROUP BY channel;
'''
run_sql(answer_sql)

共返回 3 行，当前最多显示前 10 行。
channel | user_cnt
----------------------------------------------------------------------------------------------------
ad | 3
organic | 3
referral | 2


### 题目 6：`HAVING`

查询下单次数至少为 2 次的用户，以及他们的下单总金额。

In [34]:
your_sql = '''
SELECT
    user_id,
    COUNT(*) AS order_count,
    SUM(order_amount)
FROM orders
GROUP BY user_id
HAVING COUNT(*) >= 2
'''
run_sql(your_sql)

共返回 4 行，当前最多显示前 10 行。
user_id | order_count | SUM(order_amount)
----------------------------------------------------------------------------------------------------
1 | 2 | 800.0
3 | 2 | 1100.0
4 | 2 | 1100.0
6 | 2 | 1000.0


In [40]:
your_sql='''
SELECT *
FROM(
    SELECT
    user_id,
    COUNT(*) AS order_count,
    SUM(order_amount) AS total_amt
    FROM orders
    GROUP BY user_id
) AS t
WHERE order_count >= 2;
'''
run_sql(your_sql)

共返回 4 行，当前最多显示前 10 行。
user_id | order_count | total_amt
----------------------------------------------------------------------------------------------------
1 | 2 | 800.0
3 | 2 | 1100.0
4 | 2 | 1100.0
6 | 2 | 1000.0


In [35]:
answer_sql = '''
SELECT
    user_id,
    COUNT(*) AS order_cnt,
    SUM(order_amount) AS total_amount
FROM orders
GROUP BY user_id
HAVING COUNT(*) >= 2;
'''
run_sql(answer_sql)

共返回 4 行，当前最多显示前 10 行。
user_id | order_cnt | total_amount
----------------------------------------------------------------------------------------------------
1 | 2 | 800.0
3 | 2 | 1100.0
4 | 2 | 1100.0
6 | 2 | 1000.0


### 题目 7：条件聚合

按渠道统计借款总笔数、逾期笔数、已还清笔数、进行中笔数。

In [57]:
your_sql = '''
SELECT 
    u.channel,
    l.user_id,
    COUNT(*) AS loan_count,
    SUM(
    CASE
        WHEN status = 'overdue' THEN 1
        ELSE 0
    END) as overdue,
    SUM(
    CASE
        WHEN status = 'repaid' THEN 1 ELSE 0 END) AS repaid,
    SUM(CASE WHEN status = 'pending' THEN 1 ELSE 0 END) AS pending
  FROM loans  AS L
  LEFT JOIN users AS u
  ON l.user_id = u.user_id
GROUP BY u.channel, l.user_Id;

'''
run_sql(your_sql)

共返回 7 行，当前最多显示前 10 行。
channel | user_id | loan_count | overdue | repaid | pending
----------------------------------------------------------------------------------------------------
ad | 2 | 1 | 1 | 0 | 0
ad | 3 | 1 | 1 | 0 | 0
ad | 6 | 1 | 1 | 0 | 0
organic | 1 | 2 | 0 | 2 | 0
organic | 5 | 1 | 0 | 1 | 0
organic | 8 | 1 | 0 | 0 | 1
referral | 4 | 1 | 0 | 0 | 1


In [47]:
answer_sql = '''
SELECT
    u.channel,
    COUNT(*) AS loan_cnt,
    SUM(CASE WHEN l.status = 'overdue' THEN 1 ELSE 0 END) AS overdue_cnt,
    SUM(CASE WHEN l.status = 'repaid' THEN 1 ELSE 0 END) AS repaid_cnt,
    SUM(CASE WHEN l.status = 'pending' THEN 1 ELSE 0 END) AS pending_cnt
FROM loans l
JOIN users u
    ON l.user_id = u.user_id
GROUP BY u.channel;
'''
run_sql(answer_sql)

共返回 3 行，当前最多显示前 10 行。
channel | loan_cnt | overdue_cnt | repaid_cnt | pending_cnt
----------------------------------------------------------------------------------------------------
ad | 3 | 3 | 0 | 0
organic | 4 | 0 | 3 | 1
referral | 1 | 0 | 0 | 1


### 题目 8：去重统计

按月份统计下单用户数（去重后）。

In [68]:
your_sql = '''
SELECT 
    strftime('%Y-%m' , order_date) AS order_month,
    COUNT(DISTINCT user_id) AS order_count
FROM orders
GROUP BY strftime('%Y-%m',order_date);

'''
run_sql(your_sql)

共返回 3 行，当前最多显示前 10 行。
order_month | order_count
----------------------------------------------------------------------------------------------------
2024-01 | 2
2024-02 | 3
2024-03 | 3


In [64]:
answer_sql = '''
SELECT
    strftime('%Y-%m', order_date) AS order_month,
    COUNT(DISTINCT user_id) AS distinct_order_users
FROM orders
GROUP BY strftime('%Y-%m', order_date)
ORDER BY order_month;
'''
run_sql(answer_sql)

共返回 3 行，当前最多显示前 10 行。
order_month | distinct_order_users
----------------------------------------------------------------------------------------------------
2024-01 | 2
2024-02 | 3
2024-03 | 3


## 5. JOIN 练习

### 题目 9：`INNER JOIN`

查询每笔订单对应的用户名、城市、渠道和订单金额。

In [75]:
your_sql = '''
SELECT 
    u.user_name,
    u.city,
    u.channel,
    o.order_amount
FROM users AS u
LEFT JOIN orders AS o
ON u.user_id = o.user_id
;
'''
run_sql(your_sql,limit=12)

共返回 12 行，当前最多显示前 12 行。
user_name | city | channel | order_amount
----------------------------------------------------------------------------------------------------
Alice | Shanghai | organic | 300.0
Alice | Shanghai | organic | 500.0
Bob | Beijing | ad | 800.0
Cindy | Shanghai | ad | 200.0
Cindy | Shanghai | ad | 900.0
David | Shenzhen | referral | 400.0
David | Shenzhen | referral | 700.0
Eric | Hangzhou | organic | 150.0
Frank | Suzhou | ad | 350.0
Frank | Suzhou | ad | 650.0
Grace | Beijing | referral | None
Helen | Shanghai | organic | None


In [73]:
answer_sql = '''
SELECT
    o.order_id,
    u.user_name,
    u.city,
    u.channel,
    o.order_amount
FROM orders o
INNER JOIN users u
    ON o.user_id = u.user_id;
'''
run_sql(answer_sql)

共返回 10 行，当前最多显示前 10 行。
order_id | user_name | city | channel | order_amount
----------------------------------------------------------------------------------------------------
1001 | Alice | Shanghai | organic | 300.0
1002 | Alice | Shanghai | organic | 500.0
1003 | Bob | Beijing | ad | 800.0
1004 | Cindy | Shanghai | ad | 200.0
1005 | Cindy | Shanghai | ad | 900.0
1006 | David | Shenzhen | referral | 400.0
1007 | David | Shenzhen | referral | 700.0
1008 | Eric | Hangzhou | organic | 150.0
1009 | Frank | Suzhou | ad | 650.0
1010 | Frank | Suzhou | ad | 350.0


### 题目 10：`LEFT JOIN`

查询所有用户的下单次数和总消费金额，保留没有下单的用户。

In [87]:
your_sql = '''
SELECT 
    u.user_name,
    u.user_id,
    COUNT(o.order_id) as order_count,
    COALESCE(SUM(o.order_amount),0) AS total_amount
FROM users AS u
LEFT JOIN orders AS o
ON u.user_id = o.user_id
GROUP BY u.user_id 
;
'''
run_sql(your_sql)

共返回 8 行，当前最多显示前 10 行。
user_name | user_id | order_count | total_amount
----------------------------------------------------------------------------------------------------
Alice | 1 | 2 | 800.0
Bob | 2 | 1 | 800.0
Cindy | 3 | 2 | 1100.0
David | 4 | 2 | 1100.0
Eric | 5 | 1 | 150.0
Frank | 6 | 2 | 1000.0
Grace | 7 | 0 | 0
Helen | 8 | 0 | 0


In [83]:
answer_sql = '''
SELECT
    u.user_id,
    u.user_name,
    COUNT(o.order_id) AS order_cnt,
    COALESCE(SUM(o.order_amount), 0) AS total_amount
FROM users u
LEFT JOIN orders o
    ON u.user_id = o.user_id
GROUP BY u.user_id, u.user_name
ORDER BY u.user_id;
'''
run_sql(answer_sql)

共返回 8 行，当前最多显示前 10 行。
user_id | user_name | order_cnt | total_amount
----------------------------------------------------------------------------------------------------
1 | Alice | 2 | 800.0
2 | Bob | 1 | 800.0
3 | Cindy | 2 | 1100.0
4 | David | 2 | 1100.0
5 | Eric | 1 | 150.0
6 | Frank | 2 | 1000.0
7 | Grace | 0 | 0
8 | Helen | 0 | 0


### 题目 11：多表 JOIN

使用多表关联，统计每个用户的访问次数、下单次数、借款次数。

In [95]:
your_sql = '''
WITH visit_status AS (
    SELECT user_id,
        COUNT(*) AS visit_count
    FROM visits
    GROUP BY user_id),
order_status AS (
    SELECT user_id,
        COUNT(*) AS order_count
    FROM orders
    GROUP BY user_id),
loan_status AS (
    SELECT user_id,
        COUNT(*) AS loan_count
    FROM loans
    GROUP BY user_id)
SELECT 
    u.user_id,
    u.user_name,
    COALESCE(v.visit_count,0) AS vistit_count,
    COALESCE(o.order_count,0) AS order_count,
    COALESCE(l.loan_count,0) AS loan_count
FROM users AS u
LEFT JOIN visit_status AS v
ON u.user_id = v.user_id
LEFT JOIN order_status AS o
ON u.user_id = o.user_id
LEFT JOIN loan_status AS l
ON u.user_id = l.user_id
ORDER BY u.user_id;
'''
run_sql(your_sql)

共返回 8 行，当前最多显示前 10 行。
user_id | user_name | vistit_count | order_count | loan_count
----------------------------------------------------------------------------------------------------
1 | Alice | 1 | 2 | 2
2 | Bob | 2 | 1 | 1
3 | Cindy | 1 | 2 | 1
4 | David | 1 | 2 | 1
5 | Eric | 1 | 1 | 1
6 | Frank | 2 | 2 | 1
7 | Grace | 1 | 0 | 0
8 | Helen | 1 | 0 | 1


In [89]:
answer_sql = '''
WITH visit_stats AS (
    SELECT user_id, COUNT(*) AS visit_cnt
    FROM visits
    GROUP BY user_id
),
order_stats AS (
    SELECT user_id, COUNT(*) AS order_cnt
    FROM orders
    GROUP BY user_id
),
loan_stats AS (
    SELECT user_id, COUNT(*) AS loan_cnt
    FROM loans
    GROUP BY user_id
)
SELECT
    u.user_id,
    u.user_name,
    COALESCE(v.visit_cnt, 0) AS visit_cnt,
    COALESCE(o.order_cnt, 0) AS order_cnt,
    COALESCE(l.loan_cnt, 0) AS loan_cnt
FROM users u
LEFT JOIN visit_stats v ON u.user_id = v.user_id
LEFT JOIN order_stats o ON u.user_id = o.user_id
LEFT JOIN loan_stats l ON u.user_id = l.user_id
ORDER BY u.user_id;
'''
run_sql(answer_sql)

共返回 8 行，当前最多显示前 10 行。
user_id | user_name | visit_cnt | order_cnt | loan_cnt
----------------------------------------------------------------------------------------------------
1 | Alice | 1 | 2 | 2
2 | Bob | 2 | 1 | 1
3 | Cindy | 1 | 2 | 1
4 | David | 1 | 2 | 1
5 | Eric | 1 | 1 | 1
6 | Frank | 2 | 2 | 1
7 | Grace | 1 | 0 | 0
8 | Helen | 1 | 0 | 1


## 6. 日期处理、子查询与 CTE

### 题目 12：日期处理

按月份统计订单数和 GMV。

In [101]:
your_sql = '''
SELECT 
    strftime('%Y-%m',order_date) AS order_month,
    COUNT(*) AS order_count,
    SUM(order_amount)  AS total_amount
FROM orders
GROUP BY order_month;
'''
run_sql(your_sql)

共返回 3 行，当前最多显示前 10 行。
order_month | order_count | total_amount
----------------------------------------------------------------------------------------------------
2024-01 | 2 | 1100.0
2024-02 | 3 | 1050.0
2024-03 | 5 | 2800.0


In [98]:
answer_sql = '''
SELECT
    strftime('%Y-%m', order_date) AS order_month,
    COUNT(*) AS order_cnt,
    SUM(order_amount) AS gmv
FROM orders
GROUP BY strftime('%Y-%m', order_date)
ORDER BY order_month;
'''
run_sql(answer_sql)

共返回 3 行，当前最多显示前 10 行。
order_month | order_cnt | gmv
----------------------------------------------------------------------------------------------------
2024-01 | 2 | 1100.0
2024-02 | 3 | 1050.0
2024-03 | 5 | 2800.0


### 题目 13：子查询

查询订单金额高于“该用户平均订单金额”的订单。

In [107]:
your_sql = '''
WITH user_avg AS (
    SELECT user_id,
    AVG(order_amount) AS avg_amount
    FROM orders
    GROUP BY user_id)
SELECT 
    o.order_id,
    o.user_id,
    o.order_amount
FROM orders AS o
JOIN user_avg AS ua
ON o.user_id = ua.user_id
WHERE o.order_amount > ua.avg_amount

;
'''
run_sql(your_sql)

共返回 4 行，当前最多显示前 10 行。
order_id | user_id | order_amount
----------------------------------------------------------------------------------------------------
1002 | 1 | 500.0
1005 | 3 | 900.0
1007 | 4 | 700.0
1009 | 6 | 650.0


In [104]:
answer_sql = '''
SELECT
    o.order_id,
    o.user_id,
    o.order_amount
FROM orders o
WHERE o.order_amount > (
    SELECT AVG(o2.order_amount)
    FROM orders o2
    WHERE o2.user_id = o.user_id
);
'''
run_sql(answer_sql)

共返回 4 行，当前最多显示前 10 行。
order_id | user_id | order_amount
----------------------------------------------------------------------------------------------------
1002 | 1 | 500.0
1005 | 3 | 900.0
1007 | 4 | 700.0
1009 | 6 | 650.0


### 题目 14：子查询

查询总消费最高的用户。

In [113]:
your_sql = '''
WITH user_sum AS 
    (SELECT user_id,
        SUM(order_amount) AS total_amount
    FROM orders
    GROUP BY user_id
)

SELECT  
    user_id,
    total_amount
FROM user_sum
WHERE total_amount = (
    SELECT MAX(total_amount)
    FROM user_sum)
;

'''
run_sql(your_sql)

共返回 2 行，当前最多显示前 10 行。
user_id | total_amount
----------------------------------------------------------------------------------------------------
3 | 1100.0
4 | 1100.0


In [112]:
answer_sql = '''
SELECT
    t.user_id,
    t.total_amount
FROM (
    SELECT
        user_id,
        SUM(order_amount) AS total_amount
    FROM orders
    GROUP BY user_id
) t
WHERE t.total_amount = (
    SELECT MAX(x.total_amount)
    FROM (
        SELECT SUM(order_amount) AS total_amount
        FROM orders
        GROUP BY user_id
    ) x
);
'''
run_sql(answer_sql)

共返回 2 行，当前最多显示前 10 行。
user_id | total_amount
----------------------------------------------------------------------------------------------------
3 | 1100.0
4 | 1100.0


### 题目 15：`WITH`

使用 CTE 统计每个用户的下单次数和总消费，只保留总消费大于 800 的用户。

In [118]:
your_sql = '''
WITH user_count AS (
    SELECT 
    user_id,
    COUNT(*) AS order_count,
    SUM(order_amount) AS order_sum
    FROM orders
    GROUP BY user_id)

SELECT
    user_id,
    order_count,
    order_sum
FROM user_count
WHERE order_sum > 800
GROUP BY user_id;
'''
run_sql(your_sql)

共返回 3 行，当前最多显示前 10 行。
user_id | order_count | order_sum
----------------------------------------------------------------------------------------------------
3 | 2 | 1100.0
4 | 2 | 1100.0
6 | 2 | 1000.0


In [116]:
answer_sql = '''
WITH user_order_stats AS (
    SELECT
        user_id,
        COUNT(*) AS order_cnt,
        SUM(order_amount) AS total_amount
    FROM orders
    GROUP BY user_id
)
SELECT *
FROM user_order_stats
WHERE total_amount > 800
ORDER BY total_amount DESC;
'''
run_sql(answer_sql)

共返回 3 行，当前最多显示前 10 行。
user_id | order_cnt | total_amount
----------------------------------------------------------------------------------------------------
3 | 2 | 1100.0
4 | 2 | 1100.0
6 | 2 | 1000.0


## 7. 窗口函数

### 题目 16：`ROW_NUMBER()`

先按用户统计总消费，再在每个城市内部按总消费从高到低编号，取每个城市前 2 名。

In [158]:
your_sql = '''
WITH user_spend AS (
    SELECT
    o.user_id,
    u.user_name,
    u.city,
    COALESCE(SUM(o.order_amount),0) AS total_amount
    FROM orders AS o
    LEFT JOIN users AS u
    ON o.user_id = u.user_id
    GROUP BY o.user_id,u.user_name, u.city
),
ranked AS (
    SELECT 
    *,
    ROW_NUMBER() OVER(
        PARTITION BY city
        ORDER BY total_amount DESC
    ) AS rn
    FROM user_spend
)

SELECT 
    *
FROM 
    ranked
WHERE rn <= 2
ORDER BY city,rn;
'''
run_sql(your_sql)

共返回 6 行，当前最多显示前 10 行。
user_id | user_name | city | total_amount | rn
----------------------------------------------------------------------------------------------------
2 | Bob | Beijing | 800.0 | 1
5 | Eric | Hangzhou | 150.0 | 1
3 | Cindy | Shanghai | 1100.0 | 1
1 | Alice | Shanghai | 800.0 | 2
4 | David | Shenzhen | 1100.0 | 1
6 | Frank | Suzhou | 1000.0 | 1


In [138]:
answer_sql = '''
WITH user_amount AS (
    SELECT
        u.user_id,
        u.user_name,
        u.city,
        COALESCE(SUM(o.order_amount), 0) AS total_amount
    FROM users u
    LEFT JOIN orders o
        ON u.user_id = o.user_id
    GROUP BY u.user_id, u.user_name, u.city
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY city
            ORDER BY total_amount DESC
        ) AS rn
    FROM user_amount
)
SELECT *
FROM ranked
WHERE rn <= 2
ORDER BY city, rn;
'''
run_sql(answer_sql)

共返回 7 行，当前最多显示前 10 行。
user_id | user_name | city | total_amount | rn
----------------------------------------------------------------------------------------------------
2 | Bob | Beijing | 800.0 | 1
7 | Grace | Beijing | 0 | 2
5 | Eric | Hangzhou | 150.0 | 1
3 | Cindy | Shanghai | 1100.0 | 1
1 | Alice | Shanghai | 800.0 | 2
4 | David | Shenzhen | 1100.0 | 1
6 | Frank | Suzhou | 1000.0 | 1


### 题目 17：`RANK()`

按渠道对用户总借款金额做排名。

In [170]:
your_sql = '''
WITH loans_sum AS (
    SELECT 
    l.user_id,
    u.user_name,
    u.channel,
    COALESCE(SUM(l.loan_amount),0) AS loan_sum
FROM loans AS l
LEFT JOIN users as u
ON u.user_id = l.user_id
GROUP BY u.user_id, u.channel
)
SELECT 
    *,
    RANK() OVER(
    PARTITION BY channel
    ORDER BY loan_sum DESC
    ) AS loan_rank
FROM loans_sum
ORDER BY channel, loan_rank;
'''
run_sql(your_sql)

共返回 7 行，当前最多显示前 10 行。
user_id | user_name | channel | loan_sum | loan_rank
----------------------------------------------------------------------------------------------------
6 | Frank | ad | 1600.0 | 1
2 | Bob | ad | 1500.0 | 2
3 | Cindy | ad | 1200.0 | 3
1 | Alice | organic | 3000.0 | 1
8 | Helen | organic | 1100.0 | 2
5 | Eric | organic | 900.0 | 3
4 | David | referral | 1800.0 | 1


In [161]:
answer_sql = '''
WITH user_loan_amount AS (
    SELECT
        u.user_id,
        u.user_name,
        u.channel,
        COALESCE(SUM(l.loan_amount), 0) AS total_loan_amount
    FROM users u
    LEFT JOIN loans l
        ON u.user_id = l.user_id
    GROUP BY u.user_id, u.user_name, u.channel
)
SELECT
    *,
    RANK() OVER (
        PARTITION BY channel
        ORDER BY total_loan_amount DESC
    ) AS loan_rank
FROM user_loan_amount
ORDER BY channel, loan_rank;
'''
run_sql(answer_sql)

共返回 8 行，当前最多显示前 10 行。
user_id | user_name | channel | total_loan_amount | loan_rank
----------------------------------------------------------------------------------------------------
6 | Frank | ad | 1600.0 | 1
2 | Bob | ad | 1500.0 | 2
3 | Cindy | ad | 1200.0 | 3
1 | Alice | organic | 3000.0 | 1
8 | Helen | organic | 1100.0 | 2
5 | Eric | organic | 900.0 | 3
4 | David | referral | 1800.0 | 1
7 | Grace | referral | 0 | 2


### 题目 18：`DENSE_RANK()`

先按月份和渠道统计 GMV，再在每个月内部对渠道 GMV 做稠密排名。

In [192]:
your_sql = '''
WITH channel_month_gmv AS (
    SELECT 
    strftime('%Y-%m',o.order_date) AS order_month,
    u.channel,
    COALESCE(SUM(o.order_amount),0) AS total_amount
FROM orders AS o
LEFT JOIN users AS u
    ON o.user_id = u.user_id
GROUP BY order_month, u.channel
)

SELECT 
    * ,
    DENSE_RANK() OVER(
    PARTITION BY order_month
    ORDER BY total_amount
    ) AS gmv_rank
FROM channel_month_gmv
ORDER BY order_month , gmv_rank

'''
run_sql(your_sql)

共返回 6 行，当前最多显示前 10 行。
order_month | channel | total_amount | gmv_rank
----------------------------------------------------------------------------------------------------
2024-01 | organic | 300.0 | 1
2024-01 | ad | 800.0 | 2
2024-02 | referral | 400.0 | 1
2024-02 | organic | 650.0 | 2
2024-03 | referral | 700.0 | 1
2024-03 | ad | 2100.0 | 2


In [178]:
answer_sql = '''
WITH channel_month_gmv AS (
    SELECT
        strftime('%Y-%m', o.order_date) AS order_month,
        u.channel,
        SUM(o.order_amount) AS gmv
    FROM orders o
    JOIN users u
        ON o.user_id = u.user_id
    GROUP BY strftime('%Y-%m', o.order_date), u.channel
)
SELECT
    *,
    DENSE_RANK() OVER (
        PARTITION BY order_month
        ORDER BY gmv DESC
    ) AS gmv_rank
FROM channel_month_gmv
ORDER BY order_month, gmv_rank;
'''
run_sql(answer_sql)

共返回 6 行，当前最多显示前 10 行。
order_month | channel | gmv | gmv_rank
----------------------------------------------------------------------------------------------------
2024-01 | ad | 800.0 | 1
2024-01 | organic | 300.0 | 2
2024-02 | organic | 650.0 | 1
2024-02 | referral | 400.0 | 2
2024-03 | ad | 2100.0 | 1
2024-03 | referral | 700.0 | 2


### 题目 19：`SUM() OVER()`

按月份统计 GMV，并计算累计 GMV。

In [202]:
your_sql = '''
WITH month_GMV AS (
    SELECT 
        user_id,
        strftime('%Y-%m',order_date) AS order_month,
        SUM(order_amount) AS month_amount
    FROM orders
    GROUP BY user_id, strftime('%Y-%m',order_date)
    )
SELECT 
    user_id,
    order_month,
    month_amount,
    SUM(month_amount) OVER(
    ORDER BY order_month
    ) AS total_amount
FROM
    month_GMV
ORDER BY order_month, user_id

'''
run_sql(your_sql)

共返回 8 行，当前最多显示前 10 行。
user_id | order_month | month_amount | total_amount
----------------------------------------------------------------------------------------------------
1 | 2024-01 | 300.0 | 1100.0
2 | 2024-01 | 800.0 | 1100.0
1 | 2024-02 | 500.0 | 2150.0
4 | 2024-02 | 400.0 | 2150.0
5 | 2024-02 | 150.0 | 2150.0
3 | 2024-03 | 1100.0 | 4950.0
4 | 2024-03 | 700.0 | 4950.0
6 | 2024-03 | 1000.0 | 4950.0


In [196]:
answer_sql = '''
WITH month_gmv AS (
    SELECT
        strftime('%Y-%m', order_date) AS order_month,
        SUM(order_amount) AS gmv
    FROM orders
    GROUP BY strftime('%Y-%m', order_date)
)
SELECT
    order_month,
    gmv,
    SUM(gmv) OVER (
        ORDER BY order_month
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_gmv
FROM month_gmv
ORDER BY order_month;
'''
run_sql(answer_sql)

共返回 3 行，当前最多显示前 10 行。
order_month | gmv | cumulative_gmv
----------------------------------------------------------------------------------------------------
2024-01 | 1100.0 | 1100.0
2024-02 | 1050.0 | 2150.0
2024-03 | 2800.0 | 4950.0


### 题目 20：`AVG() OVER()`

统计每个用户的总消费，并计算该用户所在渠道的人均总消费。

In [208]:
your_sql = '''
WITH user_order AS (
    SELECT
      u.user_id,
      COALESCE(SUM(o.order_amount),0) AS total_amount,
      u.channel
    FROM users AS u
    LEFT JOIN orders AS o
    ON u.user_id = o.user_id
    GROUP BY u.user_id, u.channel
)
SELECT 
    *,
    AVG(total_amount) OVER(
    PARTITION BY channel
    ) AS channel_amount
FROM user_order
ORDER BY user_id,channel
    
'''
run_sql(your_sql)

共返回 8 行，当前最多显示前 10 行。
user_id | total_amount | channel | channel_amount
----------------------------------------------------------------------------------------------------
1 | 800.0 | organic | 316.6666666666667
2 | 800.0 | ad | 966.6666666666666
3 | 1100.0 | ad | 966.6666666666666
4 | 1100.0 | referral | 550.0
5 | 150.0 | organic | 316.6666666666667
6 | 1000.0 | ad | 966.6666666666666
7 | 0 | referral | 550.0
8 | 0 | organic | 316.6666666666667


In [209]:
answer_sql = '''
WITH user_amount AS (
    SELECT
        u.user_id,
        u.user_name,
        u.channel,
        COALESCE(SUM(o.order_amount), 0) AS total_amount
    FROM users u
    LEFT JOIN orders o
        ON u.user_id = o.user_id
    GROUP BY u.user_id, u.user_name, u.channel
)
SELECT
    *,
    AVG(total_amount) OVER (PARTITION BY channel) AS channel_avg_amount
FROM user_amount
ORDER BY channel, user_id;
'''
run_sql(answer_sql)

共返回 8 行，当前最多显示前 10 行。
user_id | user_name | channel | total_amount | channel_avg_amount
----------------------------------------------------------------------------------------------------
2 | Bob | ad | 800.0 | 966.6666666666666
3 | Cindy | ad | 1100.0 | 966.6666666666666
6 | Frank | ad | 1000.0 | 966.6666666666666
1 | Alice | organic | 800.0 | 316.6666666666667
5 | Eric | organic | 150.0 | 316.6666666666667
8 | Helen | organic | 0 | 316.6666666666667
4 | David | referral | 1100.0 | 550.0
7 | Grace | referral | 0 | 550.0


## 8. 业务分析题

### 题目 21：用户分群

基于用户总消费金额做分群：
- `>= 1000` 为 `high_value`
- `500 ~ 999.99` 为 `mid_value`
- `0.01 ~ 499.99` 为 `low_value`
- 没有下单为 `no_order`

In [220]:
your_sql = '''
WITH orders_cost AS (
    SELECT 
        u.user_id,
        u.user_name,
        COALESCE(SUM(o.order_amount),0) AS total_amount
    FROM users AS u
    LEFT JOIN orders AS o
    ON u.user_id =o.user_id
    GROUP BY u.user_id, u.user_name
    )
SELECT 
    *,
    CASE
        WHEN total_amount >=1000 THEN 'high_value'
        WHEN total_amount >=500 THEN 'mid_value'
        WHEN total_amount >=0.01 THEN 'low_value'
    ELSE  'no_order'
    END AS user_group
FROM orders_cost

'''
run_sql(your_sql)

共返回 8 行，当前最多显示前 10 行。
user_id | user_name | total_amount | user_group
----------------------------------------------------------------------------------------------------
1 | Alice | 800.0 | mid_value
2 | Bob | 800.0 | mid_value
3 | Cindy | 1100.0 | high_value
4 | David | 1100.0 | high_value
5 | Eric | 150.0 | low_value
6 | Frank | 1000.0 | high_value
7 | Grace | 0 | no_order
8 | Helen | 0 | no_order


In [217]:
answer_sql = '''
WITH user_amount AS (
    SELECT
        u.user_id,
        u.user_name,
        COALESCE(SUM(o.order_amount), 0) AS total_amount
    FROM users u
    LEFT JOIN orders o
        ON u.user_id = o.user_id
    GROUP BY u.user_id, u.user_name
)
SELECT
    *,
    CASE
        WHEN total_amount >= 1000 THEN 'high_value'
        WHEN total_amount >= 500 THEN 'mid_value'
        WHEN total_amount > 0 THEN 'low_value'
        ELSE 'no_order'
    END AS user_group
FROM user_amount
ORDER BY total_amount DESC;
'''
run_sql(answer_sql)

共返回 8 行，当前最多显示前 10 行。
user_id | user_name | total_amount | user_group
----------------------------------------------------------------------------------------------------
3 | Cindy | 1100.0 | high_value
4 | David | 1100.0 | high_value
6 | Frank | 1000.0 | high_value
1 | Alice | 800.0 | mid_value
2 | Bob | 800.0 | mid_value
5 | Eric | 150.0 | low_value
7 | Grace | 0 | no_order
8 | Helen | 0 | no_order


### 题目 22：转化率

按渠道统计：访问用户数、下单用户数、转化率。

转化率 = 下单用户数 / 访问用户数。

In [224]:
your_sql = '''
WITH visit_users AS (
    SELECT 
    u.user_id,
    u.user_name,
    u.channel,
    COUNT(v.user_id) AS visit_count
FROM users AS u
LEFT JOIN visits AS v
ON u.user_id = v.user_id
GROUP BY u.user_id,u.user_name,u.channel
),
order_users AS (
    SELECT 
    u.user_id,
    COUNT(o.user_id) AS order_count
FROM users AS u
LEFT JOIN orders AS o
ON u.user_id = o.user_id
GROUP BY u.user_id
)
SELECT 
    v.channel,
    v.visit_count,
    o.order_count,
    CASE
        WHEN v.visit_count = 0 THEN 0
        ELSE ROUND(1.0 * o.order_count / v.visit_count,2)
    END AS conversion_rate
FROM visit_users AS v
LEFT JOIN order_users AS o
ON v.user_id = o.user_id


'''
run_sql(your_sql)

共返回 8 行，当前最多显示前 10 行。
channel | visit_count | order_count | conversion_rate
----------------------------------------------------------------------------------------------------
organic | 1 | 2 | 2.0
ad | 2 | 1 | 0.5
ad | 1 | 2 | 2.0
referral | 1 | 2 | 2.0
organic | 1 | 1 | 1.0
ad | 2 | 2 | 1.0
referral | 1 | 0 | 0.0
organic | 1 | 0 | 0.0


In [223]:
answer_sql = '''
WITH visit_users AS (
    SELECT
        u.channel,
        COUNT(DISTINCT v.user_id) AS visit_user_cnt
    FROM visits v
    JOIN users u
        ON v.user_id = u.user_id
    GROUP BY u.channel
),
order_users AS (
    SELECT
        u.channel,
        COUNT(DISTINCT o.user_id) AS order_user_cnt
    FROM orders o
    JOIN users u
        ON o.user_id = u.user_id
    GROUP BY u.channel
)
SELECT
    v.channel,
    v.visit_user_cnt,
    COALESCE(o.order_user_cnt, 0) AS order_user_cnt,
    ROUND(1.0 * COALESCE(o.order_user_cnt, 0) / v.visit_user_cnt, 4) AS conversion_rate
FROM visit_users v
LEFT JOIN order_users o
    ON v.channel = o.channel
ORDER BY v.channel;
'''
run_sql(answer_sql)

共返回 3 行，当前最多显示前 10 行。
channel | visit_user_cnt | order_user_cnt | conversion_rate
----------------------------------------------------------------------------------------------------
ad | 3 | 3 | 1.0
organic | 3 | 2 | 0.6667
referral | 2 | 1 | 0.5


### 题目 23：逾期率

按渠道统计：借款用户数、逾期用户数、逾期率。

逾期率 = 逾期用户数 / 借款用户数。

In [225]:
your_sql = '''
WITH loan_users AS (
    SELECT
        u.channel,
        COUNT(DISTINCT l.user_id) AS loan_user_cnt
    FROM loans l
    JOIN users u
        ON l.user_id = u.user_id
    GROUP BY u.channel
),
overdue_users AS (
    SELECT
        u.channel,
        COUNT(DISTINCT l.user_id) AS overdue_user_cnt
    FROM loans l
    JOIN users u
        ON l.user_id = u.user_id
    WHERE l.status = 'overdue'
    GROUP BY u.channel
)
SELECT
    a.channel,
    a.loan_user_cnt,
    COALESCE(b.overdue_user_cnt, 0) AS overdue_user_cnt,
    ROUND(1.0 * COALESCE(b.overdue_user_cnt, 0) / a.loan_user_cnt, 4) AS overdue_rate
FROM loan_users a
LEFT JOIN overdue_users b
    ON a.channel = b.channel
ORDER BY a.channel;
'''
run_sql(your_sql)

共返回 3 行，当前最多显示前 10 行。
channel | loan_user_cnt | overdue_user_cnt | overdue_rate
----------------------------------------------------------------------------------------------------
ad | 3 | 3 | 1.0
organic | 3 | 0 | 0.0
referral | 1 | 0 | 0.0


In [ ]:
answer_sql = '''
WITH loan_users AS (
    SELECT
        u.channel,
        COUNT(DISTINCT l.user_id) AS loan_user_cnt
    FROM loans l
    JOIN users u
        ON l.user_id = u.user_id
    GROUP BY u.channel
),
overdue_users AS (
    SELECT
        u.channel,
        COUNT(DISTINCT l.user_id) AS overdue_user_cnt
    FROM loans l
    JOIN users u
        ON l.user_id = u.user_id
    WHERE l.status = 'overdue'
    GROUP BY u.channel
)
SELECT
    a.channel,
    a.loan_user_cnt,
    COALESCE(b.overdue_user_cnt, 0) AS overdue_user_cnt,
    ROUND(1.0 * COALESCE(b.overdue_user_cnt, 0) / a.loan_user_cnt, 4) AS overdue_rate
FROM loan_users a
LEFT JOIN overdue_users b
    ON a.channel = b.channel
ORDER BY a.channel;
'''
run_sql(answer_sql)

### 题目 24：分组表现 + 时间趋势

按月份和渠道统计订单数、下单用户数、GMV，观察分组表现和时间趋势。

In [228]:
your_sql = '''
SELECT
    strftime('%Y-%m',o.order_date) AS order_month,
    u.channel,
    COUNT(o.order_id) AS order_count,
    SUM(o.order_amount) AS total_amount
FROM orders AS o
JOIN users AS u
    ON o.user_id = u.user_id
GROUP BY strftime('%Y-%m',o.order_date) , u.channel
ORDER BY order_month, u.channel;
'''
run_sql(your_sql)

共返回 6 行，当前最多显示前 10 行。
order_month | channel | order_count | total_amount
----------------------------------------------------------------------------------------------------
2024-01 | ad | 1 | 800.0
2024-01 | organic | 1 | 300.0
2024-02 | organic | 2 | 650.0
2024-02 | referral | 1 | 400.0
2024-03 | ad | 4 | 2100.0
2024-03 | referral | 1 | 700.0


In [229]:
answer_sql = '''
SELECT
    strftime('%Y-%m', o.order_date) AS order_month,
    u.channel,
    COUNT(o.order_id) AS order_cnt,
    COUNT(DISTINCT o.user_id) AS order_user_cnt,
    SUM(o.order_amount) AS gmv
FROM orders o
JOIN users u
    ON o.user_id = u.user_id
GROUP BY strftime('%Y-%m', o.order_date), u.channel
ORDER BY order_month, u.channel;
'''
run_sql(answer_sql)

共返回 6 行，当前最多显示前 10 行。
order_month | channel | order_cnt | order_user_cnt | gmv
----------------------------------------------------------------------------------------------------
2024-01 | ad | 1 | 1 | 800.0
2024-01 | organic | 1 | 1 | 300.0
2024-02 | organic | 2 | 2 | 650.0
2024-02 | referral | 1 | 1 | 400.0
2024-03 | ad | 4 | 2 | 2100.0
2024-03 | referral | 1 | 1 | 700.0


## 9. 学习提醒

如果你把这 24 题基本做顺了，已经很接近数据分析 / 风控实习常见 SQL 要求。

最建议你反复练的题型是：
- `LEFT JOIN + GROUP BY`
- 条件聚合
- 去重统计
- `CASE WHEN` 分群
- `WITH` 拆步骤
- 窗口函数做 TopN 和累计值
- 转化率 / 逾期率 / 时间趋势